# Qwen continuation dataset generator

Run with a GPU runtime. Works on Colab, Kaggle, and local Jupyter.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime first.'
print(torch.cuda.get_device_name(0))


In [ ]:
import os

try:
    import google.colab
    ENV = 'colab'
    BASE_DIR = '/content'
except ImportError:
    if os.path.exists('/kaggle'):
        ENV = 'kaggle'
        BASE_DIR = '/kaggle/working'
    else:
        ENV = 'local'
        BASE_DIR = os.getcwd()

print(f'Environment: {ENV}')
print(f'Base directory: {BASE_DIR}')


In [ ]:
REPO_URL = 'https://github.com/Teodorus730/qwen.git'
BRANCH = 'main'
PROJECT_DIR = 'qwen_continuation_dataset'
REPO_DIR = f'{BASE_DIR}/qwen'

!rm -rf {REPO_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}/{PROJECT_DIR}
!python -m pip install -q -r requirements.txt


In [ ]:
!python generate_dataset.py --config config.yaml --dry-run


## Generate

In [ ]:
!python generate_dataset.py \
  --config config.yaml \
  --dataset mixed \
  --math-ratio 0.5 \
  --mode entropy \
  --prefix-tokens 128 \
  --max-new-tokens 64 \
  --entropy-threshold 6.5 \
  --max-examples 20 \
  --preview 3 \
  --output outputs/mixed_entropy_20.jsonl \
  --overwrite


In [ ]:
!python inspect_dataset.py \
    outputs/mixed_entropy_20.jsonl \
    --show-examples 3


## Hugging Face upload

Authorise below and set `HF_REPO_ID`. `--hf-upload` and `--hf-repo-id`
override `config.yaml` for this run, so no file editing is needed on Kaggle
or after a fresh clone. Completed shards upload incrementally and resume
from the last checkpoint on restart.

In [ ]:
import os
os.environ['HF_TOKEN'] = ''  # paste token or run: !huggingface-cli login
HF_REPO_ID = 'your_username/qwen_continuation_dataset'


In [ ]:
!python generate_dataset.py \
  --config config.yaml \
  --dataset mixed \
  --math-ratio 0.5 \
  --mode entropy \
  --max-examples 50000 \
  --output outputs/train.jsonl \
  --hf-upload \
  --hf-repo-id {HF_REPO_ID}
